In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import os
 
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})
 
os.makedirs('figures', exist_ok=True)
 
DATA = '../results'
baseline = pd.read_csv(f'{DATA}/baseline/baseline.csv')
baseline.columns = baseline.columns.str.strip()
baseline['source_region'] = baseline['source_region'].str.strip()
baseline['target_region'] = baseline['target_region'].str.strip()

alphas = [1, 5, 10, 20, 30, 40, 50]
wavelet = {}
for a in alphas:
    df = pd.read_csv(f'{DATA}/wavelet/wavelet_a{a}.csv')
    df.columns = df.columns.str.strip()
    df['source_region'] = df['source_region'].str.strip()
    df['target_region'] = df['target_region'].str.strip()
    wavelet[a] = df

betas = [1, 5, 10, 15, 20, 25]
fda = {}
for b in betas:
    df = pd.read_csv(f'{DATA}/fourier/fourier_b{b}.csv')
    df.columns = df.columns.str.strip()
    df['source_region'] = df['source_region'].str.strip()
    df['target_region'] = df['target_region'].str.strip()
    fda[b] = df

def get_delta(method_df):
    m = baseline.merge(method_df, on=['source_region','target_region'], suffixes=('_base','_method'))
    m['delta'] = m['iou_method'] - m['iou_base']
    return m
 
print(f"Loaded: baseline ({len(baseline)} pairs), {len(alphas)} wavelet alphas, {len(betas)} FDA betas")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

m_wav = get_delta(wavelet[40])
d_wav = np.sort(m_wav['delta'].values)
 
m_fda = get_delta(fda[1])
d_fda = np.sort(m_fda['delta'].values)
 
x = np.arange(1, 241)
 
ax.plot(x, d_fda, color='#d62728', linewidth=1.5, label='FDA (β=0.01)', alpha=0.85)
ax.plot(x, d_wav, color='#1f77b4', linewidth=1.5, label='Wavelet (α=0.40)', alpha=0.85)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
 
ax.fill_between(x, d_fda, 0, where=(d_fda < 0), color='#d62728', alpha=0.08)
ax.fill_between(x, d_wav, 0, where=(d_wav > 0), color='#1f77b4', alpha=0.08)
 
ax.set_xlabel('Source–Target Pairs (sorted by ΔIoU)')
ax.set_ylabel('ΔIoU')
ax.set_title('Cumulative ΔIoU: Wavelet vs FDA')
ax.legend(loc='upper left')
ax.set_xlim(1, 240)
ax.set_ylim(-0.40, 0.50)
 
plt.tight_layout()
plt.savefig('../figures/cumulative_delta_iou.pdf')
plt.show()
print("Saved: ../figures/cumulative_delta_iou.pdf")


In [ ]:
m40 = get_delta(wavelet[40])
 
pivot = m40.pivot(index='source_region', columns='target_region', values='delta')
 
src_order = m40.groupby('source_region')['delta'].mean().sort_values(ascending=False).index
tgt_order = m40.groupby('target_region')['delta'].mean().sort_values(ascending=False).index
pivot = pivot.loc[src_order, tgt_order]
 
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot,
    cmap='RdBu_r',
    center=0,
    vmin=-0.30,
    vmax=0.30,
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'ΔIoU', 'shrink': 0.8},
    ax=ax,
    square=False,
)
ax.set_title('Pairwise ΔIoU Heatmap — Wavelet (α=0.40)')
ax.set_xlabel('Target Region')
ax.set_ylabel('Source Region')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
 
plt.tight_layout()
plt.savefig('../figures/heatmap_wavelet_a40.pdf')
plt.show()
print("Saved: ../figures/heatmap_wavelet_a40.pdf")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
 
alpha_vals = [a/100 for a in alphas]
mean_deltas, pct_improved, big_drops, big_gains = [], [], [], []
 
for a in alphas:
    m = get_delta(wavelet[a])
    d = m['delta']
    mean_deltas.append(d.mean())
    pct_improved.append((d > 0).mean() * 100)
    big_drops.append((d <= -0.10).sum())
    big_gains.append((d >= 0.10).sum())
 
axes[0].plot(alpha_vals, mean_deltas, 'o-', color='#1f77b4', markersize=5)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_xlabel('α')
axes[0].set_ylabel('Mean ΔIoU')
axes[0].set_title('Mean ΔIoU vs α')
 
axes[1].plot(alpha_vals, pct_improved, 's-', color='#2ca02c', markersize=5)
axes[1].axhline(50, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_xlabel('α')
axes[1].set_ylabel('% Pairs Improved')
axes[1].set_title('Improvement Rate vs α')
axes[1].set_ylim(40, 65)
 
axes[2].bar([x - 0.015 for x in alpha_vals], big_gains, width=0.025, color='#2ca02c', label='Big Gains (≥0.10)', alpha=0.8)
axes[2].bar([x + 0.015 for x in alpha_vals], big_drops, width=0.025, color='#d62728', label='Big Drops (≤−0.10)', alpha=0.8)
axes[2].set_xlabel('α')
axes[2].set_ylabel('Count')
axes[2].set_title('Big Gains vs Big Drops')
axes[2].legend(fontsize=7)
 
plt.tight_layout()
plt.savefig('../figures/alpha_sweep.pdf')
plt.show()
print("Saved: ../figures/alpha_sweep.pdf")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

beta_vals = [b/100 for b in betas]
mean_deltas, pct_improved, big_drops, big_gains = [], [], [], []

for b in betas:
    m = get_delta(fda[b])
    d = m['delta']
    mean_deltas.append(d.mean())
    pct_improved.append((d > 0).mean() * 100)
    big_drops.append((d <= -0.10).sum())
    big_gains.append((d >= 0.10).sum())

axes[0].plot(beta_vals, mean_deltas, 'o-', color='#d62728', markersize=5)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_xlabel('β')
axes[0].set_ylabel('Mean ΔIoU')
axes[0].set_title('Mean ΔIoU vs β')

axes[1].plot(beta_vals, pct_improved, 's-', color='#d62728', markersize=5)
axes[1].axhline(50, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_xlabel('β')
axes[1].set_ylabel('% Pairs Improved')
axes[1].set_title('Improvement Rate vs β')
axes[1].set_ylim(40, 55)

axes[2].bar([x - 0.008 for x in beta_vals], big_gains, width=0.012, color='#2ca02c', label='Big Gains (≥0.10)', alpha=0.8)
axes[2].bar([x + 0.008 for x in beta_vals], big_drops, width=0.012, color='#d62728', label='Big Drops (≤−0.10)', alpha=0.8)
axes[2].set_xlabel('β')
axes[2].set_ylabel('Count')
axes[2].set_title('Big Gains vs Big Drops')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig('../figures/beta_sweep.pdf')
plt.show()
print("Saved: ../figures/beta_sweep.pdf")


In [ ]:
print("="*100)
print(f"{'Method':<22} {'Mean IoU':>10} {'ΔIoU':>8} {'% Imp':>7} {'Big↑':>6} {'Big↓':>6} {'Sev↓':>6} {'Worst':>9} {'Best':>9}")
print("="*100)
print(f"{'Direct Transfer':<22} {baseline['iou'].mean():>10.4f} {'—':>8} {'—':>7} {'—':>6} {'—':>6} {'—':>6} {'—':>9} {'—':>9}")
print("-"*100)
 
for b in betas:
    m = get_delta(fda[b])
    d = m['delta']
    label = f"FDA (β={b/100:.2f})"
    print(f"{label:<22} {m['iou_method'].mean():>10.4f} {d.mean():>+8.4f} {(d>0).mean()*100:>6.1f}% {(d>=0.10).sum():>6d} {(d<=-0.10).sum():>6d} {(d<=-0.20).sum():>6d} {d.min():>+9.4f} {d.max():>+9.4f}")
 
print("-"*100)
 
for a in alphas:
    m = get_delta(wavelet[a])
    d = m['delta']
    label = f"Wavelet (α={a/100:.2f})"
    print(f"{label:<22} {m['iou_method'].mean():>10.4f} {d.mean():>+8.4f} {(d>0).mean()*100:>6.1f}% {(d>=0.10).sum():>6d} {(d<=-0.10).sum():>6d} {(d<=-0.20).sum():>6d} {d.min():>+9.4f} {d.max():>+9.4f}")
 
print("="*100)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
 
m_wav = get_delta(wavelet[40])
m_fda = get_delta(fda[1])
 
bins = np.arange(-0.40, 0.50, 0.025)
 
ax.hist(m_fda['delta'], bins=bins, alpha=0.5, color='#d62728', label='FDA (β=0.01)', edgecolor='white', linewidth=0.5)
ax.hist(m_wav['delta'], bins=bins, alpha=0.5, color='#1f77b4', label='Wavelet (α=0.40)', edgecolor='white', linewidth=0.5)
 
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(m_wav['delta'].mean(), color='#1f77b4', linewidth=1.2, linestyle='-', alpha=0.7, label=f"Wavelet mean ({m_wav['delta'].mean():+.3f})")
ax.axvline(m_fda['delta'].mean(), color='#d62728', linewidth=1.2, linestyle='-', alpha=0.7, label=f"FDA mean ({m_fda['delta'].mean():+.3f})")
 
ax.set_xlabel('ΔIoU')
ax.set_ylabel('Number of Pairs')
ax.set_title('Distribution of Adaptation Gain')
ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig('../figures/delta_distribution.pdf')
plt.show()
print("Saved: ../figures/delta_distribution.pdf")


In [ ]:
print("\n" + "="*60)
print("KEY NUMBERS FOR PAPER")
print("="*60)
 
m = get_delta(wavelet[40])
d = m['delta']
print(f"\nWavelet α=0.40:")
print(f"  Mean ΔIoU: {d.mean():+.4f}")
print(f"  % Improved: {(d>0).mean()*100:.1f}%")
print(f"  Big gains (≥0.10): {(d>=0.10).sum()}")
print(f"  Big drops (≤-0.10): {(d<=-0.10).sum()}")
print(f"  Severe drops (≤-0.20): {(d<=-0.20).sum()}")
print(f"  Worst: {d.min():+.4f}")
print(f"  Best: {d.max():+.4f}")
print(f"  Gain/Drop ratio: {(d>=0.10).sum()}/{(d<=-0.10).sum()} = {(d>=0.10).sum()/max((d<=-0.10).sum(),1):.1f}x")
 
m2 = get_delta(fda[1])
d2 = m2['delta']
print(f"\nFDA β=0.01 (best FDA):")
print(f"  Mean ΔIoU: {d2.mean():+.4f}")
print(f"  Big gains: {(d2>=0.10).sum()}")
print(f"  Big drops: {(d2<=-0.10).sum()}")
print(f"  Severe drops: {(d2<=-0.20).sum()}")
print(f"  Worst: {d2.min():+.4f}")
 
print(f"\nHeadline comparison:")
print(f"  Wavelet reduces big drops by {(d2<=-0.10).sum() - (d<=-0.10).sum()} ({(d2<=-0.10).sum()} → {(d<=-0.10).sum()})")
print(f"  Wavelet reduces severe drops by {(d2<=-0.20).sum() - (d<=-0.20).sum()} ({(d2<=-0.20).sum()} → {(d<=-0.20).sum()})")
print(f"  Wavelet produces {(d>=0.10).sum() - (d2>=0.10).sum()} more big gains ({(d2>=0.10).sum()} → {(d>=0.10).sum()})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

delta_matrix = np.array([get_delta(wavelet[a])['delta'].values for a in alphas])
best_alpha_idx = np.argmax(delta_matrix, axis=0)
best_delta = delta_matrix[best_alpha_idx, np.arange(240)]
best_alpha_val = np.array([alphas[i]/100 for i in best_alpha_idx])

d_fixed = get_delta(wavelet[40])['delta'].values

ax = axes[0]
x = np.arange(1, 241)
ax.plot(x, np.sort(get_delta(fda[1])['delta'].values), color='#d62728', linewidth=1.3, label='FDA (β=0.01)', alpha=0.8)
ax.plot(x, np.sort(d_fixed), color='#1f77b4', linewidth=1.3, label='Wavelet (α=0.40)', alpha=0.8)
ax.plot(x, np.sort(best_delta), color='#ff7f0e', linewidth=1.3, label='Wavelet (oracle α)', alpha=0.8, linestyle='--')
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Source–Target Pairs (sorted)')
ax.set_ylabel('ΔIoU')
ax.set_title('Oracle α vs Fixed α')
ax.legend(fontsize=7)
ax.set_xlim(1, 240)

ax = axes[1]
from collections import Counter
counts = Counter(best_alpha_val)
alpha_labels = sorted(counts.keys())
alpha_counts = [counts[a] for a in alpha_labels]
ax.bar([str(a) for a in alpha_labels], alpha_counts, color='#1f77b4', alpha=0.8, edgecolor='white')
ax.set_xlabel('Optimal α')
ax.set_ylabel('Number of Pairs')
ax.set_title('Distribution of Optimal α')

plt.tight_layout()
plt.savefig('../figures/oracle_alpha.pdf')
plt.show()

print(f"\nOracle α selection:")
print(f"  Mean ΔIoU: {best_delta.mean():+.4f}")
print(f"  % Improved: {(best_delta > 0).mean()*100:.1f}%")
print(f"  Big gains (≥0.10): {(best_delta >= 0.10).sum()}")
print(f"  Big drops (≤-0.10): {(best_delta <= -0.10).sum()}")
print(f"  Severe drops (≤-0.20): {(best_delta <= -0.20).sum()}")
print(f"Saved: ../figures/oracle_alpha.pdf")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

betas = [1, 5, 10, 15, 20, 25]
records = {}
for b in betas:
    df = pd.read_csv(f'../results/edge_analysis/edge_metrics_b{b}.csv')
    df.columns = df.columns.str.strip()
    p99 = df['fda_hf_ratio'].quantile(0.99)
    df = df[df['fda_hf_ratio'] < p99].copy()   # remove extreme outliers
    records[b] = df

def agg(records, col):
    return (
        [records[b][col].mean() for b in betas],
        [records[b][col].std()  for b in betas],
    )

edge_mean,  edge_std  = agg(records, 'fda_edge_sim')
hf_mean,    hf_std    = agg(records, 'fda_hf_ratio')
lh_mean,    lh_std    = agg(records, 'fda_LH_corr')
hl_mean,    hl_std    = agg(records, 'fda_HL_corr')
hh_mean,    hh_std    = agg(records, 'fda_HH_corr')

ref = pd.read_csv('../results/edge_analysis/edge_metrics_b15.csv')
ref.columns = ref.columns.str.strip()
WAV_EDGE = ref['wav_edge_sim'].mean()
WAV_HF   = ref['wav_hf_ratio'].mean()
WAV_LH   = ref['wav_LH_corr'].mean()
WAV_HL   = ref['wav_HL_corr'].mean()
WAV_HH   = ref['wav_HH_corr'].mean()

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8,
})

FDA_COLOR = '#C94040'
WAV_COLOR = '#4C8DBF'
BAND_ALPHA = 0.15
GRID_COLOR = '#eeeeee'
REF_COLOR  = '#888888'

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
fig.subplots_adjust(wspace=0.38, left=0.07, right=0.97, top=0.88, bottom=0.15)

x = np.array(betas)

def plot_sweep(ax, mean, std, ylabel, title, ylim,
               ref_val=None, ref_label=None, ideal_line=None):
    mean, std = np.array(mean), np.array(std)

    ax.fill_between(x, mean - std, mean + std,
                    color=FDA_COLOR, alpha=BAND_ALPHA, zorder=2)
    ax.plot(x, mean, color=FDA_COLOR, linewidth=2,
            marker='o', markersize=5, zorder=3, label='FDA')

    if ref_val is not None:
        ax.axhline(ref_val, color=WAV_COLOR, linewidth=1.6,
                   linestyle='--', zorder=4, label=ref_label or 'Wavelet (α=0.40)')

    if ideal_line:
        ax.axhline(ideal_line, color=REF_COLOR, linewidth=0.85,
                   linestyle=':', zorder=1)

    ax.set_xlim(0, 27)
    ax.set_xticks(betas)
    ax.set_xticklabels([f'β={b}' for b in betas], rotation=30, ha='right')
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.7)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

plot_sweep(axes[0], edge_mean, edge_std,
           ylabel='Edge Similarity',
           title='Edge Preservation vs β',
           ylim=(0.75, 1.02),
           ref_val=WAV_EDGE,
           ideal_line=1.0)

plot_sweep(axes[1], hf_mean, hf_std,
           ylabel='HF Energy Ratio',
           title='HF Energy Distortion vs β',
           ylim=(0.85, 2.2),
           ref_val=WAV_HF,
           ideal_line=1.0)

ax = axes[2]
subband_cfg = [
    (lh_mean, lh_std, WAV_LH, 'LH', '#e07b39'),
    (hl_mean, hl_std, WAV_HL, 'HL', '#9b59b6'),
    (hh_mean, hh_std, WAV_HH, 'HH', '#27ae60'),
]
for mean, std, wav_ref, label, color in subband_cfg:
    mean, std = np.array(mean), np.array(std)
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=BAND_ALPHA, zorder=2)
    ax.plot(x, mean, color=color, linewidth=2,
            marker='o', markersize=5, zorder=3, label=f'FDA {label}')
    ax.axhline(wav_ref, color=color, linewidth=1.2,
               linestyle='--', alpha=0.6, zorder=1)

ax.axhline(1.0, color=REF_COLOR, linewidth=0.85, linestyle=':', zorder=1)
ax.set_xlim(0, 27)
ax.set_xticks(betas)
ax.set_xticklabels([f'β={b}' for b in betas], rotation=30, ha='right')
all_vals = lh_mean + hl_mean + hh_mean
ax.set_ylim(min(all_vals) - 0.03, 1.02)
ax.set_ylabel('Correlation with Original')
ax.set_title('Subband Preservation vs β')
ax.set_axisbelow(True)
ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.7)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))
ax.legend(fontsize=7, frameon=True, framealpha=0.85,
          edgecolor='#cccccc', loc='lower left', ncol=1)

for ax in axes[:2]:
    ax.legend(fontsize=7.5, frameon=True, framealpha=0.85,
              edgecolor='#cccccc', loc='upper right' if ax is axes[1] else 'lower left')

plt.tight_layout()
plt.savefig('../figures/edge_distortion_sweep.pdf', bbox_inches='tight')
plt.show()
print("Saved: ../figures/edge_distortion_sweep.pdf")